In [0]:
# Installing MissForest and ydata-profiling
%pip install missforest
%pip install lightgbm
%pip install tqdm
%pip install phik==0.11.1
%pip install ydata-profiling==4.1.2

In [0]:
%restart_python

In [0]:
# Importing Librarires
import warnings
import pandas as pd
import numpy as np
from missforest import MissForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from ydata_profiling import ProfileReport
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [0]:
# Loading the dataset and setting target variable
df = spark.table("workspace.default.credit_card_clients").toPandas()

y = df['default payment next month']
x = df.drop(['default payment next month', 'ID'], axis=1)


In [0]:

#train/test split with a ratio of 85/15
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.15, random_state= 67, stratify= y)

In [0]:
profile = ProfileReport(x_train, title="Credit Card Default EDA", explorative=True)
profile.to_file("docs/EDA.html")

In [0]:
x_train[x_train.duplicated()]

In [0]:
# There are 43 duplicated entries but all of them are just inactive users so they could just be people with the same demographic, however there is one observation at index 0 that is active and has 3 duplicates, so they will be dropped
duplicates_mask = ~x_train.duplicated()
x_train = x_train[duplicates_mask]
y_train = y_train[duplicates_mask]

In [0]:
# According to the documentation of the dataset
# Education (1 = graduate school; 2 = university; 3 = high school; 4 = others).
# Marriage(1 = married; 2 = single; 3 = others).
# But there are a few observations that are not documented, and therefore will be considerd as Nulls
# We will use MissForest to figure out their real value
undocumented = {
    'EDUCATION': {0: np.nan, 5: np.nan, 6: np.nan},
    'MARRIAGE': {0: np.nan}
    }
x_train.replace(undocumented, inplace= True)
x_test.replace(undocumented, inplace= True)

In [0]:
# MissForest imputation
cat_vars = ['SEX', 'EDUCATION', 'MARRIAGE']

missforest_imputer = MissForest(clf= RandomForestClassifier(n_estimators=100, max_depth= 10, random_state= 67), categorical= cat_vars)

x_train = pd.DataFrame(missforest_imputer.fit_transform(x_train), columns=x.columns, index= x_train.index)
x_test = pd.DataFrame(missforest_imputer.transform(x_test), columns=x.columns, index= x_test.index)

In [0]:
# Re-combining X and y into one dataframe to save
x_train['default_payment_next_month'] = y_train
x_train['split'] = 'train' 

x_test['default_payment_next_month'] = y_test
x_test['split'] = 'test' 

master_df = pd.concat([x_train, x_test], axis=0)
master_df.insert(0, 'ID', df['ID'])

spark.createDataFrame(master_df).write.mode("overwrite").saveAsTable("workspace.default.clean_model_data")